# **Prediksi Praktikum 08 - IF5100 Pemrograman untuk Data Analitik** — KUNCI JAWABAN

Nama : ______________________

NIM  : ______________________

### Aturan / Rules

- Kerjakan di **Google Colab** (runtime CPU). Jalankan sel **berurutan** dari atas ke bawah, karena tiap soal memakai hasil soal sebelumnya.
- Library yang boleh dipakai: `numpy`, `pandas`, `matplotlib`, `seaborn`.
- **Jangan ubah** nama fungsi, parameter, dan kode uji. Output grafik tidak harus 100% sama, tetapi nilai yang dikembalikan fungsi harus benar.
- Setiap grafik wajib punya **judul, label sumbu, dan legend** (jika ada lebih dari satu seri).

## Studi kasus: Kopi Senja

Kamu adalah analis data di jaringan kedai **Kopi Senja** yang punya 4 cabang: Bandung, Jakarta, Surabaya, dan Yogyakarta.
Manajer memberikan data transaksi **September–Oktober 2026**. Data ini diambil dari 4 sistem kasir yang berbeda,
sehingga **kotor**: penulisan nama cabang tidak seragam, harga tersimpan sebagai teks (`"Rp 27.000"`, `"?"`),
ada umur pelanggan yang mustahil, rating kosong, transaksi ganda, dan kolom yang isinya sama semua.

Tugasmu: **membersihkan data**, lalu menjawab pertanyaan bisnis manajer dengan **visualisasi** yang tepat.

*You are a data analyst at the Kopi Senja coffee chain (4 branches). The September–October 2026 transaction data is messy. Clean it, then answer the manager's business questions with the right visualizations.*

| Kolom | Isi | Tipe |
|---|---|---|
| `id_transaksi` | ID unik transaksi | identifier |
| `tanggal` | tanggal transaksi (teks `YYYY-MM-DD`) | temporal |
| `cabang` | nama cabang (penulisan tidak seragam) | kategorikal nominal |
| `kategori` | Kopi / Non-Kopi / Makanan | kategorikal nominal |
| `harga` | harga satuan, **teks**: `27000`, `Rp 27.000`, atau `?` | seharusnya numerik |
| `qty` | jumlah item | numerik diskret |
| `metode_bayar` | QRIS / Tunai / Kartu | kategorikal nominal |
| `umur_pelanggan` | umur pelanggan (ada NaN dan nilai mustahil) | numerik |
| `durasi_tunggu` | lama menunggu pesanan (menit) | numerik kontinu |
| `rating` | rating 1–5 (ada NaN) | ordinal |
| `pajak_persen` | persentase pajak | konstan |

### Setup dataset
Jalankan sel ini lebih dulu. **Jangan diubah.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 120)


def buat_dataset(seed=8, n=640):
    """Data transaksi Kopi Senja (Sep-Okt 2026) yang sengaja dibuat kotor. JANGAN DIUBAH."""
    rng = np.random.default_rng(seed)
    tanggal = pd.Timestamp('2026-09-01') + pd.to_timedelta(rng.integers(0, 61, n), unit='D')
    cabang = rng.choice(['Bandung', 'Jakarta', 'Surabaya', 'Yogyakarta'], n, p=[.30, .34, .21, .15])
    kategori = rng.choice(['Kopi', 'Non-Kopi', 'Makanan'], n, p=[.50, .22, .28])
    dasar = pd.Series({'Kopi': 27000, 'Non-Kopi': 23000, 'Makanan': 33000})
    harga = dasar[kategori].to_numpy() + rng.integers(-4, 5, n) * 1000
    akhir_pekan = tanggal.dayofweek >= 5
    qty = rng.integers(1, 4, n) + akhir_pekan * rng.integers(0, 3, n)
    metode = rng.choice(['QRIS', 'Tunai', 'Kartu'], n, p=[.55, .25, .20])
    umur = np.clip(rng.normal(27, 7, n), 16, 65).round()
    durasi = (rng.gamma(2.2, 3.2, n) + np.where(cabang == 'Jakarta', 4, 0)).round(1)
    rating = np.clip(np.round(5.2 - 0.13 * durasi + rng.normal(0, .6, n)), 1, 5)

    df = pd.DataFrame({
        'id_transaksi': [f'TRX{i:04d}' for i in range(1, n + 1)],
        'tanggal': tanggal.strftime('%Y-%m-%d'), 'cabang': cabang, 'kategori': kategori,
        'harga': harga.astype(str), 'qty': qty, 'metode_bayar': metode,
        'umur_pelanggan': umur, 'durasi_tunggu': durasi, 'rating': rating, 'pajak_persen': 11,
    })
    # ---- membuat data kotor ----
    i = rng.choice(n, 70, replace=False)
    df.loc[i[:30], 'cabang'] = df.loc[i[:30], 'cabang'].str.lower()
    df.loc[i[30:50], 'cabang'] = df.loc[i[30:50], 'cabang'].str.upper()
    df.loc[i[50:], 'cabang'] = ' ' + df.loc[i[50:], 'cabang'] + ' '
    j = rng.choice(n, 160, replace=False)
    df.loc[j[:140], 'harga'] = ['Rp ' + f'{int(h):,}'.replace(',', '.') for h in df.loc[j[:140], 'harga']]
    df.loc[j[140:], 'harga'] = '?'
    k = rng.choice(n, 40, replace=False)
    df.loc[k[:6], 'umur_pelanggan'] = [999, 0, 150, 999, 2, 120]
    df.loc[k[6:], 'umur_pelanggan'] = np.nan
    df.loc[rng.choice(n, 45, replace=False), 'rating'] = np.nan
    dup = df.sample(18, random_state=seed)
    return pd.concat([df, dup]).sample(frac=1, random_state=seed).reset_index(drop=True)


df_raw = buat_dataset()
print(df_raw.shape)
print(df_raw.head(8).to_string())

---
## SOAL 1 — Pembersihan data (20 poin)

Implementasikan `bersihkanData(df_raw)` yang mengembalikan **DataFrame baru** yang sudah bersih (jangan ubah `df_raw`).
Lakukan langkah berikut **berurutan**:

1. **Seragamkan** `cabang`: hapus spasi di awal/akhir dan ubah ke *Title Case* (`" jakarta"` → `"Jakarta"`).
2. Ubah `harga` menjadi **numerik**: buang `"Rp"`, spasi, dan titik ribuan; nilai `"?"` menjadi `NaN`.
3. **Hapus baris duplikat**, lalu reset index.
4. Isi `harga` yang kosong dengan **median harga per kategori**.
5. `umur_pelanggan` di luar rentang **15–80** dianggap tidak valid → jadikan `NaN`, lalu isi semua `NaN` dengan **median**.
6. Isi `rating` yang kosong dengan **median**.
7. Ubah `tanggal` menjadi `datetime`.
8. **Buang kolom konstan** (hanya punya 1 nilai unik).
9. Tambahkan kolom `total = harga * qty` dan `akhir_pekan` (`True` jika Sabtu/Minggu).

*Implement bersihkanData(df_raw): standardize branch names, convert price text to numbers, drop duplicates, impute price by category median, invalidate ages outside 15–80 and impute with the median, impute rating, parse dates, drop constant columns, add total and akhir_pekan.*

In [ ]:
def bersihkanData(df_raw):
    df = df_raw.copy()
    # 1) seragamkan teks cabang
    df['cabang'] = df['cabang'].str.strip().str.title()
    # 2) harga teks -> angka ('Rp 27.000' -> 27000, '?' -> NaN)
    df['harga'] = pd.to_numeric(df['harga'].str.replace(r'[^0-9]', '', regex=True), errors='coerce')
    # 3) duplikat (setelah teks seragam, sebelum imputasi)
    df = df.drop_duplicates().reset_index(drop=True)
    # 4) harga kosong -> median per kategori
    df['harga'] = df['harga'].fillna(df.groupby('kategori')['harga'].transform('median'))
    # 5) umur tidak valid -> NaN -> median
    df.loc[~df['umur_pelanggan'].between(15, 80), 'umur_pelanggan'] = np.nan
    df['umur_pelanggan'] = df['umur_pelanggan'].fillna(df['umur_pelanggan'].median())
    # 6) rating kosong -> median
    df['rating'] = df['rating'].fillna(df['rating'].median())
    # 7) tanggal -> datetime
    df['tanggal'] = pd.to_datetime(df['tanggal'])
    # 8) buang kolom konstan
    df = df.drop(columns=df.columns[df.nunique() <= 1])
    # 9) fitur baru
    df['total'] = df['harga'] * df['qty']
    df['akhir_pekan'] = df['tanggal'].dt.dayofweek >= 5
    return df


# ---- KODE UJI (jangan diubah) ----
df = bersihkanData(df_raw)
print('Ukuran awal :', df_raw.shape)
print('Ukuran akhir:', df.shape)
print('Cabang      :', sorted(df['cabang'].unique()))
print('Missing     :', int(df.isna().sum().sum()))
print('Rentang umur:', df['umur_pelanggan'].min(), '-', df['umur_pelanggan'].max())
print(df.dtypes)

**Penjelasan:**

- **Urutan penting.** Teks cabang diseragamkan dulu agar `" Jakarta "` dan `"jakarta"` dianggap sama, lalu duplikat dibuang **sebelum** imputasi supaya median tidak dihitung dari baris ganda.
- `str.replace(r'[^0-9]', '')` membuang semua karakter selain angka. `"?"` menjadi string kosong, lalu `to_numeric(errors='coerce')` mengubahnya ke `NaN`.
- Harga diisi **median per kategori** karena harga Makanan, Kopi, dan Non-Kopi berbeda jauh. Median keseluruhan akan memberi harga yang salah untuk tiap kategori.
- Umur 0, 2, 120, 150, dan 999 mustahil secara domain, jadi diperlakukan sebagai *missing*, bukan dibuang barisnya (transaksinya tetap valid).
- `pajak_persen` bernilai 11 di semua baris (variansi 0) sehingga tidak informatif dan dibuang.
- Hasil: **658 → 640 baris** (18 duplikat dibuang), 0 missing value, 4 nama cabang.

---
## SOAL 2 — Penjualan per cabang (15 poin)

Manajer bertanya: **cabang mana yang penjualannya paling besar?**

1. Implementasikan `penjualanPerCabang(df)` yang mengembalikan DataFrame ber-index `cabang` dengan kolom
   `total_penjualan` (jumlah `total`), `jumlah_transaksi`, dan `rata_per_transaksi`, **terurut menurun** berdasarkan `total_penjualan`.
2. Buat **bar chart horizontal** total penjualan (dalam juta rupiah) dengan label nilai di ujung batang.

*Implement penjualanPerCabang(df) returning total sales, number of transactions and average per transaction by branch (sorted descending), then draw a labelled horizontal bar chart.*

In [ ]:
def penjualanPerCabang(df):
    hasil = (df.groupby('cabang')
               .agg(total_penjualan=('total', 'sum'), jumlah_transaksi=('id_transaksi', 'count'))
               .sort_values('total_penjualan', ascending=False))
    hasil['rata_per_transaksi'] = (hasil['total_penjualan'] / hasil['jumlah_transaksi']).round()
    return hasil


ringkas = penjualanPerCabang(df)
print(ringkas)

urut = ringkas.sort_values('total_penjualan')          # terbesar di atas
fig, ax = plt.subplots(figsize=(8, 4))
bar = ax.barh(urut.index, urut['total_penjualan'] / 1e6, color='#1f77b4')
ax.bar_label(bar, fmt='%.1f jt', padding=3)
ax.set_title('Total Penjualan per Cabang (Sep-Okt 2026)', weight='bold')
ax.set_xlabel('total penjualan (juta Rp)')
ax.set_xlim(0, urut['total_penjualan'].max() / 1e6 * 1.15)
plt.tight_layout()
plt.show()

**Penjelasan:**

- **Bar chart** dipilih karena membandingkan nilai antar **kategori** memakai *position/length*, channel paling akurat. Diurutkan agar ranking langsung terbaca, dan horizontal agar nama cabang mudah dibaca.
- Sumbu x dimulai dari **0**. Panjang batang adalah nilainya, jadi memotong sumbu akan melebih-lebihkan selisih.
- **Jakarta** tertinggi dengan Rp 15.8 juta dari 243 transaksi; **Yogyakarta** terendah (Rp 4.8 juta).

---
## SOAL 3 — Distribusi durasi tunggu (15 poin)

Pelanggan mengeluh soal antrean. Analisis **durasi tunggu**:

1. Implementasikan `statistikDurasi(df)` yang mengembalikan dictionary `{'mean', 'median', 'skew', 'n_outlier'}`
   (mean & median dibulatkan 2 desimal, skew 3 desimal, outlier memakai **aturan IQR 1,5×**).
2. Buat 2 panel berdampingan: **histogram** `durasi_tunggu` dengan garis vertikal mean & median, dan **boxplot** `durasi_tunggu` per cabang.

*Implement statistikDurasi(df) (mean, median, skewness, IQR outlier count) and plot a histogram with mean/median lines next to a boxplot by branch.*

In [ ]:
def statistikDurasi(df):
    s = df['durasi_tunggu']
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_out = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    return {'mean': round(float(s.mean()), 2), 'median': round(float(s.median()), 2),
            'skew': round(float(s.skew()), 3), 'n_outlier': int(n_out)}


stat = statistikDurasi(df)
print(stat)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].hist(df['durasi_tunggu'], bins=25, color='#1f77b4', edgecolor='white')
axs[0].axvline(stat['mean'], color='red', ls='--', label=f"mean {stat['mean']}")
axs[0].axvline(stat['median'], color='green', ls='--', label=f"median {stat['median']}")
axs[0].set(title='Distribusi Durasi Tunggu', xlabel='menit', ylabel='jumlah transaksi')
axs[0].legend()
sns.boxplot(data=df, x='cabang', y='durasi_tunggu', color='#9ecae1', ax=axs[1])
axs[1].set(title='Durasi Tunggu per Cabang', xlabel='', ylabel='menit')
plt.tight_layout()
plt.show()

**Penjelasan:**

- Skewness **1.198** (positif) → distribusi **miring ke kanan**: kebanyakan pesanan selesai cepat, tetapi ada ekor panjang pesanan yang sangat lama. Karena itu **mean (8.49) > median (7.55)**.
- Ada **18** transaksi outlier menurut aturan IQR. Ini kemungkinan **nyata** (antrean panjang), bukan galat input, jadi tidak dibuang.
- Boxplot memperlihatkan **Jakarta** paling lama (median 9.6 menit vs 5.5–6.2 menit di cabang lain): seluruh kotaknya bergeser ke atas, jadi yang lambat bukan hanya beberapa pesanan, tetapi layanannya secara umum.
- Histogram menjawab "bagaimana bentuk sebarannya", sedangkan boxplot memudahkan **membandingkan sebaran antar grup** sekaligus menandai outlier.

---
## SOAL 4 — Komposisi kategori produk (15 poin)

Apakah komposisi produk yang terjual **berbeda antar cabang**?

1. Implementasikan `komposisiKategori(df)` yang mengembalikan tabel **proporsi** transaksi tiap `kategori` per `cabang`
   (baris = cabang, kolom = kategori, **setiap baris berjumlah 1**, dibulatkan 3 desimal).
2. Visualisasikan dengan **100% stacked bar chart** horizontal, lengkap dengan label persentase di tiap segmen.

*Implement komposisiKategori(df) returning the share of each product category per branch (rows sum to 1), then draw a labelled 100% stacked bar chart.*

In [ ]:
def komposisiKategori(df):
    return pd.crosstab(df['cabang'], df['kategori'], normalize='index').round(3)


komp = komposisiKategori(df)
print(komp)

ax = komp.plot(kind='barh', stacked=True, figsize=(9, 4), width=.7,
               color=['#8c564b', '#ff7f0e', '#2ca02c'])
for c in ax.containers:
    ax.bar_label(c, labels=[f'{v:.0%}' for v in c.datavalues], label_type='center',
                 color='white', fontsize=8)
ax.set(title='Komposisi Kategori Produk per Cabang', xlabel='proporsi transaksi', ylabel='')
ax.legend(title='kategori', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

**Penjelasan:**

- `pd.crosstab(..., normalize='index')` langsung menghitung proporsi per baris (bagian terhadap keseluruhan tiap cabang).
- **100% stacked bar** dipakai karena yang ditanya adalah **komposisi (part-to-whole) yang dibandingkan antar cabang**. Empat pie chart berdampingan sulit dibandingkan karena mata harus menilai sudut, channel yang kurang akurat.
- Kopi mendominasi di semua cabang (49%–58%). **Surabaya** paling tinggi porsi kopinya (58%); tiga cabang lainnya hampir identik (sekitar 49%–50%).

---
## SOAL 5 — Tren penjualan harian (15 poin)

Manajer ingin melihat **tren penjualan harian** dan pengaruh **akhir pekan**.

1. Implementasikan `penjualanHarian(df)` yang mengembalikan **Series** total penjualan per tanggal. Index harus berupa
   tanggal **lengkap setiap hari** (hari tanpa transaksi bernilai 0).
2. Buat **line chart** penjualan harian (juta Rp) beserta **rata-rata bergerak 7 hari**, dan tandai titik akhir pekan.
3. Cetak rata-rata penjualan per hari untuk hari kerja vs akhir pekan.

*Implement penjualanHarian(df) returning a daily sales Series with every calendar day, plot it with a 7-day moving average and highlight weekends, then compare weekday vs weekend averages.*

In [ ]:
def penjualanHarian(df):
    harian = df.groupby('tanggal')['total'].sum()
    return harian.asfreq('D', fill_value=0)      # tanggal tanpa transaksi -> 0


harian = penjualanHarian(df)
print(harian.describe().round(0))
ma7 = harian.rolling(7).mean()
weekend = harian[harian.index.dayofweek >= 5]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(harian.index, harian / 1e6, color='#1f77b4', alpha=.45, marker='o', ms=3, label='harian')
ax.plot(ma7.index, ma7 / 1e6, color='#ff7f0e', lw=2.2, label='rata-rata bergerak 7 hari')
ax.scatter(weekend.index, weekend / 1e6, color='#d62728', s=18, zorder=3, label='akhir pekan')
ax.set(title='Penjualan Harian Kopi Senja', ylabel='juta Rp')
ax.legend()
plt.tight_layout()
plt.show()

per_hari = harian.groupby(harian.index.dayofweek >= 5).mean() / 1e6
print(per_hari.rename({False: 'hari kerja', True: 'akhir pekan'}).round(2))

**Penjelasan:**

- `asfreq('D', fill_value=0)` memastikan sumbu waktu kontinu. Tanpa itu, hari tanpa transaksi hilang dan garis langsung "melompat".
- Titik merah (akhir pekan) umumnya berada di **puncak** (82% hari akhir pekan di atas median hari kerja). Rata-rata penjualan akhir pekan **Rp 0.87 juta/hari** vs hari kerja **Rp 0.59 juta/hari** (sekitar **1.5×**).
- **Rata-rata bergerak 7 hari** meratakan pola mingguan sehingga tren jangka panjang terlihat: bergerak di kisaran **Rp 0.57–0.78 juta/hari** tanpa tren naik atau turun yang jelas.
- Line chart tidak wajib mulai dari 0, karena yang dibaca adalah pola naik-turun.

---
## SOAL 6 — Durasi tunggu vs rating (10 poin)

Apakah **menunggu lama membuat rating turun**?

1. Implementasikan `korelasiDurasiRating(df)` yang mengembalikan koefisien korelasi Pearson antara `durasi_tunggu` dan `rating` (3 desimal).
2. Buat 2 panel: **scatter plot** durasi vs rating dengan garis regresi (beri *jitter* karena rating berupa bilangan bulat), dan **heatmap korelasi** kolom numerik.

*Implement korelasiDurasiRating(df) (Pearson r, 3 decimals), then plot a jittered scatter with a regression line and a correlation heatmap of the numeric columns.*

In [ ]:
def korelasiDurasiRating(df):
    return round(float(df['durasi_tunggu'].corr(df['rating'])), 3)


r = korelasiDurasiRating(df)
print('r(durasi_tunggu, rating) =', r)

fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
sns.regplot(data=df, x='durasi_tunggu', y='rating', y_jitter=.15, ax=axs[0],
            scatter_kws={'alpha': .35, 's': 15}, line_kws={'color': 'red'})
axs[0].set(title=f'Durasi Tunggu vs Rating (r = {r})', xlabel='durasi tunggu (menit)')
num = df[['harga', 'qty', 'total', 'umur_pelanggan', 'durasi_tunggu', 'rating']]
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, ax=axs[1])
axs[1].set_title('Matriks Korelasi')
plt.tight_layout()
plt.show()

**Penjelasan:**

- r = **-0.681** → korelasi **negatif kuat**: makin lama menunggu, rating cenderung makin rendah.
- Garis regresi menurun jelas. *Jitter* dipakai karena rating hanya bernilai 1–5, sehingga tanpa jitter titik-titiknya bertumpuk.
- Di heatmap, `total` berkorelasi sangat kuat dengan `qty` (0.93) tetapi hanya 0.39 dengan `harga`: `total = harga × qty`, dan variasi qty (1–5 item) jauh lebih besar daripada variasi harga. `umur_pelanggan` hampir tidak berkorelasi dengan apa pun.
- **Korelasi bukan sebab-akibat.** Tetapi bersama temuan Soal 3 (Jakarta menunggu paling lama), ini memberi dasar untuk menambah barista di Jakarta, lalu dievaluasi ulang.

---
## SOAL 7 — Perbaiki kode visualisasi (10 poin)

Kode di bawah ditulis mengikuti contoh lama dan mengandung **4 kesalahan** (error, peringatan *deprecated*, atau hasil yang menyesatkan).
Temukan dan perbaiki semuanya, lalu jelaskan tiap perbaikan.

*The code below contains 4 mistakes (errors, deprecated calls or misleading output). Find and fix them, then explain each fix.*

In [ ]:
# KODE YANG SALAH (untuk dibaca, jangan dijalankan apa adanya)
# fig, axs = plt.subplots(1, 3, figsize=(15, 4))
# sns.distplot(df['umur_pelanggan'], kde=False, bins=10, ax=axs[0])
# axs[0].set_title('Umur Pelanggan')
#
# axs[1].hist(df['durasi_tunggu'], bins=20, normed=True)
# axs[1].set_title('Durasi Tunggu')
#
# bayar = df['metode_bayar'].value_counts()
# warna = plt.get_cmap('Set2')(np.linspace(0, 1, 2))
# axs[2].pie(bayar, labels=bayar.index, autopct='%1.0f%%', colors=warna)
# axs[2].set_title('Metode Bayar')
# plt.show()
#
# print(df.corr())

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(df['umur_pelanggan'], bins=10, ax=axs[0])                 # (1) distplot -> histplot
axs[0].set_title('Umur Pelanggan')

axs[1].hist(df['durasi_tunggu'], bins=20, density=True)                # (2) normed -> density
axs[1].set(title='Durasi Tunggu', ylabel='density')

bayar = df['metode_bayar'].value_counts()
warna = plt.get_cmap('Set2')(np.linspace(0, 1, len(bayar)))           # (3) jumlah warna = jumlah kategori
axs[2].pie(bayar, labels=bayar.index, autopct='%1.0f%%', colors=warna)
axs[2].set_title('Metode Bayar')
plt.tight_layout()
plt.show()

print(df.corr(numeric_only=True).round(2))                             # (4) numeric_only

**Penjelasan:**

1. `sns.distplot` sudah **deprecated** sejak seaborn 0.11 → ganti `sns.histplot` (tanpa KDE adalah default).
2. `normed=True` sudah **dihapus** dari matplotlib (`AttributeError`) → ganti `density=True`.
3. Hanya 2 warna untuk 3 kategori, sehingga **warna berulang** dan dua irisan tampak sama (menyesatkan) → jumlah warna = `len(bayar)`.
4. `df.corr()` **error** di pandas ≥ 2 karena ada kolom teks (`cabang`, `kategori`, …) → `df.corr(numeric_only=True)`.

---
## Tugas Analisis
Jawab di sel teks di bawah setiap pertanyaan.

*Answer in the text cell below each question.*

### 1. Berdasarkan Soal 2, apakah cabang dengan total penjualan tertinggi juga yang nilai rata-rata per transaksinya tertinggi? Apa artinya?

*Based on Problem 2, does the branch with the highest total sales also have the highest average value per transaction? What does that imply?*

**Jawaban:** Tidak harus. **Jakarta** unggul karena **jumlah transaksinya paling banyak** (243). Rata-rata per transaksi semua cabang hampir sama (Rp 60,374–65,500), sehingga perbedaan total penjualan lebih ditentukan oleh **volume pelanggan**, bukan besarnya belanja per orang. Untuk menaikkan penjualan cabang kecil, fokusnya menarik lebih banyak pelanggan.

### 2. Mengapa nilai `durasi_tunggu` dan `umur_pelanggan` yang kosong sebaiknya diisi dengan median, bukan mean?

*Why should missing values be filled with the median rather than the mean?*

**Jawaban:** Karena distribusinya **tidak simetris atau mengandung outlier**. `durasi_tunggu` miring ke kanan (skew 1.198), dan `umur_pelanggan` awalnya berisi nilai mustahil seperti 999. Mean tertarik ke arah nilai ekstrem, sedangkan **median robust** (tidak terpengaruh outlier), sehingga lebih mewakili nilai tipikal.

### 3. Manajer meminta 4 pie chart (satu per cabang) untuk komposisi kategori. Apakah itu pilihan terbaik? Jelaskan dengan konsep visual encoding.

*The manager asks for 4 pie charts (one per branch). Is that the best choice? Explain using visual-encoding concepts.*

**Jawaban:** Kurang tepat. Pie chart mengandalkan **angle/area**, yang menurut ranking akurasi persepsi berada di bawah **position/length**. Membandingkan irisan antar 4 pie yang terpisah juga sulit (yang dibandingkan harus berdekatan dan memakai skala sama). **100% stacked bar** menaruh semua cabang pada satu sumbu yang sama sehingga perbedaan proporsi langsung terlihat.

### 4. Apa fungsi rata-rata bergerak 7 hari pada Soal 5, dan mengapa jendelanya 7?

*What is the purpose of the 7-day moving average in Problem 5, and why a window of 7?*

**Jawaban:** Penjualan punya **pola mingguan** (puncak tiap akhir pekan). Jendela **7 hari** selalu memuat tepat satu siklus minggu (5 hari kerja + 2 hari akhir pekan), sehingga efek akhir pekan teredam dan **tren jangka panjang** terlihat. Jendela lain (misalnya 5) akan memuat jumlah akhir pekan yang berubah-ubah, sehingga garisnya tetap bergelombang.

### 5. Berdasarkan Soal 3 dan Soal 6, rekomendasi apa yang kamu berikan ke manajer? Sertakan bukti dari grafik.

*Based on Problems 3 and 6, what would you recommend to the manager? Cite evidence from the charts.*

**Jawaban:** Prioritaskan **mempercepat layanan di cabang Jakarta**. Buktinya: median durasi tunggu Jakarta 9.6 menit vs 5.5–6.2 menit di cabang lain (boxplot Soal 3), dan durasi tunggu berkorelasi negatif dengan rating (r = -0.681, Soal 6). Karena korelasi bukan sebab-akibat, rekomendasi ini perlu diuji, misalnya menambah barista pada jam sibuk, lalu bandingkan rating sebelum dan sesudahnya.

---
## Soal Teori

### 1. Apa beda visualisasi *exploratory* dan *explanatory*? Beri contoh dari studi kasus Kopi Senja.

**Jawaban:** **Exploratory** dipakai analis untuk menemukan pola: mencoba banyak grafik, misalnya histogram semua kolom atau heatmap korelasi. **Explanatory** dipakai untuk menyampaikan satu pesan ke audiens, misalnya satu bar chart "Jakarta paling lama menunggu" untuk rapat manajer.

### 2. Urutkan channel berikut dari yang paling akurat untuk data kuantitatif: *area, color, position, angle, length*.

**Jawaban:** **position → length → angle → area → color** (Spoerri; Mackinlay). Karena itu bar chart (position/length) lebih akurat daripada pie (angle) atau bubble (area).

### 3. Mengapa color hue (merah, hijau, biru) kurang tepat untuk data ordinal seperti rating 1–5? Apa alternatifnya?

**Jawaban:** Hue **tidak punya urutan alami** di persepsi manusia, sehingga pembaca tidak tahu apakah hijau lebih tinggi dari biru. Untuk ordinal, pakai **saturation/lightness** (gradasi terang–gelap satu warna, misalnya `cmap='Blues'`) atau position.

### 4. Mengapa sumbu y bar chart wajib dimulai dari 0, sedangkan line chart tidak?

**Jawaban:** Pada bar chart, nilai dibaca dari **panjang batang**; memotong sumbu membuat selisih tampak berlipat (melanggar *lie factor*). Line chart dibaca dari **posisi dan kemiringan** (pola naik-turun), jadi sumbu boleh di-zoom ke rentang data.

### 5. Pilih grafik yang paling tepat: (a) sebaran umur pelanggan; (b) tren penjualan 2 bulan; (c) hubungan harga dan qty; (d) perpindahan pelanggan dari kanal iklan ke cabang; (e) pangsa penjualan per cabang lalu per kategori.

**Jawaban:** (a) **histogram/boxplot**, (b) **line chart**, (c) **scatter plot**, (d) **sankey diagram** (aliran source → target), (e) **tree map** (hierarki cabang → kategori, luas = penjualan).

### 6. Apa itu *redundant encoding*? Beri contoh.

**Jawaban:** Satu variabel dikodekan di **lebih dari satu channel** agar lebih cepat dan akurat dibaca, misalnya kategori dibedakan dengan **warna dan bentuk marker** sekaligus (`sns.scatterplot(hue='cabang', style='cabang')`). Ini juga membantu pembaca buta warna.

### 7. Apa pelajaran dari *Anscombe's quartet*?

**Jawaban:** Empat dataset dengan mean, variansi, korelasi, dan garis regresi yang **hampir identik** ternyata berbentuk sangat berbeda saat diplot (linear, melengkung, ada outlier, satu titik ekstrem). Jadi **selalu visualisasikan data**, jangan hanya mengandalkan statistik ringkasan.